In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
fin_data_master = pd.read_csv('financial_data.csv') 

In [6]:
prolong_master = pd.read_csv('prolongations.csv')

In [7]:
fin_data = fin_data_master.copy()
display(fin_data.head(3))

,id,Причина дубля,Ноябрь 2022,Декабрь 2022,Январь 2023,Февраль 2023,Март 2023,Апрель 2023,Май 2023,Июнь 2023,Июль 2023,Август 2023,Сентябрь 2023,Октябрь 2023,Ноябрь 2023,Декабрь 2023,Январь 2024,Февраль 2024,Account
0,42,NaN,"36 220,00",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
1,657,первая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович
2,657,вторая часть оплаты,стоп,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Васильев Артем Александрович


In [8]:
prolong = prolong_master.copy()
display(prolong.head(3))

,id,month,AM
0,42,ноябрь 2022,Васильев Артем Александрович
1,453,ноябрь 2022,Васильев Артем Александрович
2,548,ноябрь 2022,Михайлов Андрей Сергеевич


In [9]:
fin_data = fin_data.drop(['Причина дубля'], axis=1)

In [10]:
end_data = fin_data.copy()

for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x in ['end', 'стоп'] else False)
    
end_data = end_data.groupby(by='id', as_index=False).sum()

for col in end_data.columns[1:17]:
    end_data[col] = end_data[col].apply(lambda x: True if x == 1 else False)

In [11]:
zero_data = fin_data.copy()

for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 'в ноль' else False)
    
zero_data = zero_data.groupby(by='id', as_index=False).sum()

for col in zero_data.columns[1:17]:
    zero_data[col] = zero_data[col].apply(lambda x: True if x == 1 else False)

In [12]:
pay_data = fin_data.copy()

for col in pay_data.columns[1:17]:
    pay_data[col] = pay_data[col].apply(lambda x: np.nan if x in ['end', 'стоп', 'в ноль'] else x)
    pay_data[col] = pay_data[col].fillna('0,00')
    pay_data[col] = pay_data[col].apply(lambda x: x[:-3])
    pay_data[col] = pay_data[col].apply(lambda x: x.replace('\xa0', ''))
    pay_data[col] = pay_data[col].apply(lambda x: x.replace(',', ''))
    pay_data[col] = pay_data[col].astype(int)
    
pay_data = pay_data.groupby(by='id', as_index=False).sum()

def manager_back(txt):
    name_list = txt.split(' ')
    if len(name_list) <= 3 :
        return txt
    else:    
        t = len(name_list[0])
        name = f'{name_list[0]} {name_list[1]} {name_list[2]}'
        name = name[:-t]
    return name

pay_data['Account'] = pay_data['Account'].apply(manager_back)

In [13]:
# prolong МОДИФИКАЦИЯ НАЗВАНИЯ МЕСЯЦА
def first_symbol(txt):
    frst = txt[0]
    frst = frst.upper()
    txt = frst + txt[1:]
    return txt

prolong['month'] = prolong['month'].apply(first_symbol)


In [ ]:
# НЕ ЗАПУСКАТЬ, объединил через JOIN очень коряво
general_table = pay_data.join(
    other = prolong,
    how = 'left',
    on = 'id',
    lsuffix = '_l',
    rsuffix = '_r'
)

In [ ]:
# НЕ ЗАПУСКАТЬ, объединил через MERGE напрасно
general_table = pd.merge(
    left = pay_data,
    right = prolong,
    how = 'left',
    left_on = 'id',
    right_on = 'id'
)

general_table = general_table.drop(['AM'], axis=1)

In [ ]:
# НЕ ЗАПУСКАТЬ, напрасно созданный close_month_data
# close_month_data, версия созданная из general_table
close_month_data = general_table.copy()

for col in close_month_data.columns[1:17] :
    for string in close_month_data.index :
        if close_month_data.loc[string, 'month'] == col:
            close_month_data.at[string, col] = True
        else: 
            close_month_data.at[string, col] = False
            
close_month_data = close_month_data.groupby(by='id', as_index=False).sum()

for col in close_month_data.columns[1:17]:
    close_month_data[col] = close_month_data[col].apply(lambda x: True if x == 1 else False)
    
#close_month_data = close_month_data.drop(['Account', 'month'], axis=1)

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2254743313.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = False
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2254743313.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'True' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = True
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2254743313.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  close_month_data.at[string, col] = False
C:\

In [15]:
# ЛАСТ-МОНФС ИЗ ПЭЙ-ДАТА
last_months_data = pay_data.copy()

end_project_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        if id_project in end_project_list :
            last_months_data.iat[string, col] = 0
        else:
            # по id проекта находим строку с информацией о последних месяцах этого проекта
            end_index = end_data[end_data['id'] == id_project].index
            if (end_data.iloc[end_index, col]).bool() == True :
                end_project_list.append(id_project)
                last_months_data.iat[string, col] = 0
                continue
            close_month_index = close_month_data[close_month_data['id'] == id_project].index
            if (close_month_data.iloc[close_month_index, col]).bool() == False :
                last_months_data.iat[string, col] = 0
            else:
                zero_index = zero_data[zero_data['id'] == id_project].index
                if (zero_data.iloc[zero_index, col]).bool() == True :
                    pay_index = pay_data[pay_data['id'] == id_project].index
                    last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]

C:\Users\azott\AppData\Local\Temp\ipykernel_8164\2073179320.py:15: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (end_data.iloc[end_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\2073179320.py:20: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (close_month_data.iloc[close_month_index, col]).bool() == False :
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\2073179320.py:24: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (zero_data.iloc[zero_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\2073179320.py:26: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]


СОСТАВЛЯЮ ТАБЛИЦЫ СРАЗУ

In [53]:
# 1-Я ПРОЛОНГАЦ по менеджерам
# создадим словать, где <key-порядковый по range> и <value-название месяца>, т.е. {1: 'Ноябрь 2022', 2: 'Декабрь 2022' ... и т.д.}
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]
    
###################################################################################################

# создадим first_prolongation как копию pay_data, чтобы таблица содержала сумму отгрузок по менеджерам после окончаний проектов
first_prolongation = pay_data.copy()

# в Ноябре 2022 (месяц 1) сразу проставим 1-ую пролонгацию как ноль
first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Февраля 2024 (месяц 16)
for col in range(2, 17) :
    # и пройдёмся циклом по всем индексам first_prolongation
    for string in first_prolongation.index :
        # если в соответствующих "координатах" проекта и предыдущего месяца (таблица close_month_data) будет False
        if close_month_data.iloc[string, (col - 1)] == False:
            # то в текущих строке проекта и месяце (найденном по заранее созданному словарю dict_months) ставим ноль
            first_prolongation.at[string, dict_months[col]] = 0
            # если же в этом месяце проект закрывается (True), то значение останется на месте

###################################################################################################

#------# для подсчёта количества пролонгированных проектов в 1-ый месяц создадим таблицу
#------first_cnt_prolong = first_prolongation.copy()

# сгруппируем данные в first_prolongation по менеджерам, считая в месяцах суммы
first_prolongation = first_prolongation.groupby(by='Account', as_index=True).sum()

# удалим лишние столбцы
first_prolongation = first_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
#--------first_prolongation = first_prolongation.drop([0], axis=0)

###################################################################################################

# для подсчёта 1-го коэффициента сделаем таблицу first_coef, как копию таблицы first_prolongation
first_coef = first_prolongation.copy()

# и создадим словарь coef_sum_dict, в котором по ключу месяца значением будет сумма отгрузки за прошлый месяц, 
# т.е. {'Январь 2023': <сумма отгрузки за Декабрь 2022>, 'Февраль 2023': <сумма отгрузки за Январь 2023> ... и т.д.}
coef_sum_dict = {}

# для заполнения словаря coef_sum_dict пройдёмся циклом по номерам месяцев, 
# от Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13) - они будут предыдущими месяцами
for last_month in range(2, 14) :
    # зная номер предыдущего месяца, будем находить столбец следующего (целевого) месяца
    goal_month = last_months_data.columns[last_month + 1]
    # затем находить сумму отгрузки за предыдущий месяц coef_sum
    coef_sum = last_months_data[last_months_data.columns[last_month]].sum()
    # и по ключу целевого месяца goal_month в словарь coef_sum_dict 
    # будем заносить значение суммы отгрузки за предыдущий месяц coef_sum
    coef_sum_dict[goal_month] = coef_sum

# пройдёмся циклом по каждому месяцу таблицы first_coef
for col in first_coef.columns[0:13] :
    # и пройдёмся циклом по каждому индексу first_coef (в роли индексов у нас уже ФИО менеджеров)
    for string in first_coef.index:
        # в итоге для каждого менеджера в каждом месяце рассчитываем отношение суммы пролонгации у конкретного менеджера
        # к сумме отгрузки всех закрывающихся в предудыщем месяце проектов, округляя до 3-х знаков после запятой
        first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_8164\759300205.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.186' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\759300205.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.257' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\759300205.py:63: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.247' has dtype incompatible with int64, p

In [ ]:
first_coef.to_excel(excel_writer = 'C:\\IT\Коэффициенты пролонгации по менеджерам.xlsx', sheet_name = '1-ый коэффициент', index=True)

In [61]:
# 2-Я ПРОЛОНГАЦ по менеджерам
# создадим словать, где <key-порядковый по range> и <value-название месяца>, т.е. {1: 'Ноябрь 2022', 2: 'Декабрь 2022' ... и т.д.}
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]
    
###################################################################################################

# создадим second_prolongation как копию pay_data, чтобы таблица содержала сумму отгрузок по менеджерам после окончаний проектов
second_prolongation = pay_data.copy()

# в Ноябре 2022 (месяц 1) и Декабре 2022 (месяц 2) сразу проставим 2-ую пролонгацию как ноль
second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Января 2023 (месяц 3) до Февраля 2024 (месяц 16)
for col in range(3, 17) :
    # и пройдёмся циклом по всем индексам second_prolongation
    for string in second_prolongation.index :
        # если в соответствующих "координатах" проекта и два месяца назад (таблица close_month_data) будет False,
        # а также в таблице pay_data месяц назад будет ноль
        if close_month_data.iloc[string, (col - 2)] == True and pay_data.iloc[string, (col - 1)] == 0:
            # то в текущих строке проекта и месяце оставляем всё, как было
            continue
        else:
            # иначе, если оба условия не выполняются, то в текущих строке проекта и месяце 
            # (найденном по заранее созданному словарю dict_months) ставим ноль
            second_prolongation.at[string, dict_months[col]] = 0
            
###################################################################################################

#------# для подсчёта количества пролонгированных проектов в 1-ый месяц создадим таблицу
#------first_cnt_prolong = first_prolongation.copy()

# сгруппируем данные в second_prolongation по менеджерам, считая в месяцах суммы
second_prolongation = second_prolongation.groupby(by='Account', as_index=True).sum()

# удалим лишние столбцы
second_prolongation = second_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
#--------first_prolongation = first_prolongation.drop([0], axis=0)

###################################################################################################

# для подсчёта 2-го коэффициента сделаем таблицу second_coef, как копию таблицы second_prolongation
second_coef = second_prolongation.copy()

# и создадим словарь coef_sum_dict_2, в котором по ключу месяца значением будет сумма отгрузки за два месяца назад 
# при условии отсутствия отгрузок месяц назад, т.е. 
# {'Январь 2023': <сумма отгрузки заканчивающихся проектов за Ноябрь 2022, если нет отгрузок в Декабре 2022>,
# 'Февраль 2023': <сумма отгрузки заканчивающихся проектов за Декабрь 2022, если нет отгрузок в Январее 2023> ... и т.д.}
coef_sum_dict_2 = {}

# для заполнения словаря coef_sum_dict_2 пройдёмся циклом по номерам месяцев, 
# от Ноября 2022 (месяц 1) до Ноября 2023 (месяц 13) - они будут позапрошлыми месяцами
for before_last_month in range(1, 14) :
    # зная номер позапрошлого месяца, будем находить столбец текущего (целевого) месяца
    goal_month = last_months_data.columns[before_last_month + 2]
    # затем проверять является ли позапрошлый месяц завершающим для проекта (по таблице close_month_data)
    # и были ли отгрузки в предыдущем месяце по нулям (по таблице pay_data),
    # предварительно для подсчёта создадим переменную общей суммы искомой отгрузки sum_for_goal
    sum_for_goal = 0
    for string in close_month_data.index :
        if close_month_data.iloc[string, before_last_month] == True and pay_data.iloc[string, before_last_month + 1] == 0:
            sum_for_goal += pay_data.iloc[string, before_last_month]
    # затем эту сумму sum_for_goal вносим как значение в словарь coef_sum_dict_2, 
    # где ключом будет целевой месяц goal_month
    coef_sum_dict_2[goal_month] = sum_for_goal

# пройдёмся циклом по каждому месяцу таблицы second_coef
for col in second_coef.columns[0:13] :
    # и пройдёмся циклом по каждому индексу second_coef (в роли индексов у нас уже ФИО менеджеров)
    for string in second_coef.index:
        # в итоге для каждого менеджера в каждом месяце рассчитываем отношение суммы пролонгации у конкретного менеджера
        # к сумме отгрузки всех закрывающихся в предудыщем месяце проектов, округляя до 3-х знаков после запятой
        second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict_2[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_8164\1909075927.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.135' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict_2[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\1909075927.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.058' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict_2[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_8164\1909075927.py:75: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.089' has dtype incompatible wi

In [99]:
second_coef.to_excel(excel_writer = 'C:\\IT\ 2-ой коэффициент пролонгации по менеджерам.xlsx', sheet_name = '2-ой коэффициент', index=True)

In [81]:
# для DataFrame к пролонгации 1-ый месяц
# для расчёта количества проектов к пролонгации в 1-ый месяц сделаем копию close_month_data
manag_projects_for_first = close_month_data.copy()

# и в новой таблице manag_projects_for_first все булевы значения заменим на нули
for col in manag_projects_for_first.columns[1:17]:
    manag_projects_for_first[col] = manag_projects_for_first[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13)
for col in range(2, 14) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True
        if close_month_data.iloc[string, col] == True:
            # то в таблице manag_projects_for_first на следующий месяц ставим 1
            manag_projects_for_first.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_for_first = manag_projects_for_first.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_for_first['сумма к 1-ой пролонгации'] = manag_projects_for_first.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов к 1-ой пролонгации
manag_projects_for_first = manag_projects_for_first.groupby('Account')['сумма к 1-ой пролонгации'].sum()


In [90]:
# для DataFrame к пролонгации 2-ой месяц
# для расчёта количества проектов к пролонгации во 2-ой месяц сделаем копию close_month_data
manag_projects_for_second = close_month_data.copy()

# и в новой таблице manag_projects_for_second все булевы значения заменим на нули
for col in manag_projects_for_second.columns[1:17]:
    manag_projects_for_second[col] = manag_projects_for_second[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Октября 2023 (месяц 12)
for col in range(1, 13) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и в текущих строчке и в следующем месяце в таблице pay_data будет 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] == 0 :
            # то в таблице all_projects_for_first на два месяца вперёд ставим 1
            manag_projects_for_second.iat[string, (col + 2)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_for_second = manag_projects_for_second.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_for_second['сумма ко 2-ой пролонгации'] = manag_projects_for_second.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов ко 2-ой пролонгации
manag_projects_for_second = manag_projects_for_second.groupby('Account')['сумма ко 2-ой пролонгации'].sum()

In [88]:
# для DataFrame пролонгировано в 1-ый месяц
# для расчёта количества проектов, пролонгированных в 1-ый месяц, сделаем копию close_month_data
manag_projects_first_prolong = close_month_data.copy()

# и в новой таблице manag_projects_first_prolong все булевы значения заменим на нули
for col in manag_projects_first_prolong.columns[1:17]:
    manag_projects_first_prolong[col] = manag_projects_first_prolong[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Декабря 2022 (месяц 2) до Ноября 2023 (месяц 13)
for col in range(2, 14) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и если в текущей строчке и следующем месяце в таблице pay_data будет не 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] != 0 :
            # то в таблице manag_projects_first_prolong на следующий месяц ставим 1
            manag_projects_first_prolong.iat[string, (col + 1)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_first_prolong = manag_projects_first_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_first_prolong['сумма пролонгировано в 1-ый месяц'] = manag_projects_first_prolong.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов к 1-ой пролонгации
manag_projects_first_prolong = manag_projects_first_prolong.groupby('Account')['сумма пролонгировано в 1-ый месяц'].sum()

In [92]:
# для DataFrame пролонгировано во 2-ый месяц
# для расчёта количества проектов, пролонгированных во 2-ой месяц, сделаем копию close_month_data
manag_projects_second_prolong = close_month_data.copy()

# и в новой таблице manag_projects_second_prolong все булевы значения заменим на нули
for col in manag_projects_second_prolong.columns[1:17]:
    manag_projects_second_prolong[col] = manag_projects_second_prolong[col].apply(lambda x: 0)

# пройдёмся циклом по порядку номера месяца, начиная с Ноября 2022 (месяц 1) до Октября 2023 (месяц 12)
for col in range(1, 13) :
    # и пройдёмся циклом по всем индексам close_month_data
    for string in close_month_data.index :
        # если в текущих строчке и месяце в таблице close_month_data будет True 
        # и если в текущей строчке и следующем месяце в таблице pay_data будет 0, 
        # но ещё месяцем позже в таблице pay_data будет не 0
        if close_month_data.iloc[string, col] == True and pay_data.iloc[string, (col + 1)] == 0 and pay_data.iloc[string, (col + 2)] != 0:
            # то в таблице manag_projects_second_prolong на два месяца вперёд ставим 1
            manag_projects_second_prolong.iat[string, (col + 2)] = 1

# удалим лишние столбцы с id проекта и месяцами         
manag_projects_second_prolong = manag_projects_second_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

# посчитаем сумму по каждой строчке
manag_projects_second_prolong['сумма пролонгировано во 2-ой месяц'] = manag_projects_second_prolong.sum(numeric_only=True, axis=1)

# а затем группируем по менеджерам с подсчётом общей суммы проектов к 1-ой пролонгации
manag_projects_second_prolong = manag_projects_second_prolong.groupby('Account')['сумма пролонгировано во 2-ой месяц'].sum()

In [ ]:
# для DataFrame 1-ый коэффициент по менеджерам
# 
first_coef_manag = first_coef.copy()
first_coef_manag['1-ый коэффициент среднее за год'] = round( first_coef_manag.mean(numeric_only=True, axis=1) , 3)
for col in first_coef_manag.columns[0:12]:
    first_coef_manag = first_coef_manag.drop([col], axis=1)

In [62]:
# для DataFrame 2-ой коэффициент по менеджерам
#
second_coef_manag = second_coef.copy()
second_coef_manag['2-ой коэффициент среднее за год'] = round( second_coef_manag.mean(numeric_only=True, axis=1) , 3)
for col in second_coef_manag.columns[0:12]:
    second_coef_manag = second_coef_manag.drop([col], axis=1)

In [93]:
managers_for_year = pd.DataFrame({
    'Менеджер': manag_projects_for_first.index,
    'к пролонгации в 1-ый месяц': list(manag_projects_for_first),
    'пролонгировано в 1-ый месяц': list(manag_projects_first_prolong),
    'коэффициент 1-ой пролонгации': list(first_coef_manag['1-ый коэффициент среднее за год']),
    'к пролонгации во 2-ой месяц': list(manag_projects_for_second),
    'пролонгировано во 2-ой месяц': list(manag_projects_second_prolong),
    'коэффициент 2-ой пролонгации': list(second_coef_manag['2-ой коэффициент среднее за год'])
    })

In [100]:
managers_for_year.to_excel(excel_writer = 'C:\\IT\Менеджеры за год.xlsx', sheet_name = 'менеджеры за год', index=True)

In [ ]:
all_dept = pd.DataFrame(
    {'Месяц': columns=}
    {'к пролонгации 1-ый месяц': all_projects_for_first},
    {'пролонгировано 1-ый месяц': all_projects_first_prolong},
    {'коэффициент 1-ой пролонгации': first_coef_dept},
    {'к пролонгации 2-ой месяц': all_projects_for_second},
    {'пролонгировано 2-ой месяц': ll_projects_second_prolong},
    {'коэффициент 2-ой пролонгации': second_coef_dept},
)

In [67]:
# 1-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА
dict_months = {}
for n in range(1, 17):
    dict_months[n] = pay_data.columns[n]

first_prolongation = pay_data.copy()

first_prolongation['Ноябрь 2022'] = first_prolongation['Ноябрь 2022'].apply(lambda x: 0)

for col in range(2, 17) :
    for string in first_prolongation.index :
        if close_month_data.iloc[string, (col - 1)] == False:
            first_prolongation.at[string, dict_months[col]] = 0

# для подсчёта количества пролонгированных проектов в 1-ый месяц создадим таблицу
first_cnt_prolong = first_prolongation.copy()
            
first_prolongation = first_prolongation.groupby(by='Account', as_index=False).sum()

first_prolongation = first_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
#first_prolongation = first_prolongation.drop([0], axis=0)

In [100]:
# 1-ОЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
first_cnt_project = close_month_data.copy()

end_project_list = []
for col in range(1, 17):
    for string in first_cnt_project.index :
        id_project = first_cnt_project.loc[string, 'id']
        if id_project in end_project_list :
            first_cnt_project.iat[string, col] = False
        else:
            if end_data.iloc[string, col] == True :
                end_project_list.append(id_project)
                first_cnt_project.iloc[string, col] = False

first_cnt_project['Account'] = first_cnt_project['Account'].apply(manager_back)

first_cnt_project = first_cnt_project.drop(['id', 'month'], axis=1)

first_cnt_project = first_cnt_project.groupby(by='Account', as_index=False).sum()

first_cnt_project_copy = first_cnt_project.copy()
last_month = 'Декабрь 2022'
for month in first_cnt_project.columns[3:14] :
    first_cnt_project[month] = first_cnt_project_copy[last_month]
    last_month = month
    
first_cnt_project = first_cnt_project.drop(['Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

KeyError: "['month'] not found in axis"

In [15]:
# 1-ОЕ КОЛ-ВО ПРОЛОНГИРОВАННЫХ
# first_cnt_prolong из ячейки [# 1-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА]

for col in first_cnt_prolong.columns[3:15] :
    first_cnt_prolong[col] = first_cnt_prolong[col].apply(lambda x: False if x == 0 else True)

first_cnt_prolong['Account'] = first_cnt_prolong['Account'].apply(manager_back)

first_cnt_prolong = first_cnt_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

first_cnt_prolong = first_cnt_prolong.groupby(by='Account', as_index=False).sum()

In [16]:
# 2-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА
second_prolongation = pay_data.copy()

second_prolongation['Ноябрь 2022'] = second_prolongation['Ноябрь 2022'].apply(lambda x: 0)
second_prolongation['Декабрь 2022'] = second_prolongation['Декабрь 2022'].apply(lambda x: 0)

for col in range(3, 17) :
    for string in second_prolongation.index :
        if close_month_data.iloc[string, (col - 2)] == False:
            second_prolongation.at[string, dict_months[col]] = 0
            
# для подсчёта количества пролонгированных проектов во 2-ой месяц создадим таблицу
second_cnt_prolong = second_prolongation.copy()

second_prolongation = second_prolongation.groupby(by='Account', as_index=False).sum()

second_prolongation = second_prolongation.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)
second_prolongation = second_prolongation.drop([0], axis=0)

In [17]:
# 2-ОЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
second_cnt_project = close_month_data.copy()

end_project_list = []
for col in range(1, 17):
    for string in second_cnt_project.index :
        id_project = second_cnt_project.loc[string, 'id']
        if id_project in end_project_list :
            second_cnt_project.iat[string, col] = False
        else:
            if end_data.iloc[string, col] == True :
                end_project_list.append(id_project)
                second_cnt_project.iloc[string, col] = False

second_cnt_project['Account'] = second_cnt_project['Account'].apply(manager_back)

second_cnt_project = second_cnt_project.drop(['id', 'month'], axis=1)

second_cnt_project = second_cnt_project.groupby(by='Account', as_index=False).sum()

second_cnt_project_copy = second_cnt_project.copy()
double_last_month = 'Ноябрь 2022'
last_month = 'Декабрь 2022'
for month in second_cnt_project.columns[3:14] :
    second_cnt_project[month] = second_cnt_project_copy[double_last_month]
    double_last_month = last_month
    last_month = month
    
second_cnt_project = second_cnt_project.drop(['Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

In [18]:
# 2-ОЕ КОЛ-ВО ПРОЛОНГИРОВАННЫХ
# second_cnt_prolong из ячейки [# 2-Я ПРОЛОНГАЦ ИЗ ПЭЙ-ДАТА]

for col in second_cnt_prolong.columns[3:15] :
    second_cnt_prolong[col] = second_cnt_prolong[col].apply(lambda x: False if x == 0 else True)

second_cnt_prolong['Account'] = second_cnt_prolong['Account'].apply(manager_back)

second_cnt_prolong = second_cnt_prolong.drop(['id', 'Ноябрь 2022', 'Декабрь 2022', 'Январь 2024', 'Февраль 2024'], axis=1)

second_cnt_prolong = second_cnt_prolong.groupby(by='Account', as_index=False).sum()

In [61]:
# ОБЩЕЕ КОЛ-ВО ПРОЕКТОВ К ПРОЛОНГАЦИИ
all_projects_for_first = {}

for month in first_cnt_project.columns[1:13] :
    all_projects_for_first[month] = first_cnt_project[month].sum()
    
all_projects_for_second = {}

for month in second_cnt_project.columns[1:13] :
    all_projects_for_second[month] = second_cnt_project[month].sum()

In [29]:
# ОБЩЕЕ КОЛ-ВО ПРОЛОНГИРОВАННЫХ
all_projects_first_prolong = {}

for month in first_cnt_prolong.columns[1:13] :
    all_projects_first_prolong[month] = first_cnt_prolong[month].sum()
    
all_projects_second_prolong = {}

for month in second_cnt_prolong.columns[1:13] :
    all_projects_second_prolong[month] = second_cnt_prolong[month].sum()

In [ ]:
# пока закрыл, рабочая версия ЛАСТ-МОНФС, перенёс ее выше
# ЛАСТ-МОНФС ИЗ ПЭЙ-ДАТА
last_months_data = pay_data.copy()

end_project_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        if id_project in end_project_list :
            last_months_data.iat[string, col] = 0
        else:
            # по id проекта находим строку с информацией о последних месяцах этого проекта
            end_index = end_data[end_data['id'] == id_project].index
            if (end_data.iloc[end_index, col]).bool() == True :
                end_project_list.append(id_project)
                last_months_data.iat[string, col] = 0
                continue
            close_month_index = close_month_data[close_month_data['id'] == id_project].index
            if (close_month_data.iloc[close_month_index, col]).bool() == False :
                last_months_data.iat[string, col] = 0
            else:
                zero_index = zero_data[zero_data['id'] == id_project].index
                if (zero_data.iloc[zero_index, col]).bool() == True :
                    pay_index = pay_data[pay_data['id'] == id_project].index
                    last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:15: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (end_data.iloc[end_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:20: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (close_month_data.iloc[close_month_index, col]).bool() == False :
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:24: FutureWarning: Series.bool is now deprecated and will be removed in future version of pandas
  if (zero_data.iloc[zero_index, col]).bool() == True :
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\2073179320.py:26: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]


In [ ]:
# СУММАРНЫЙ КОЭФ ПРОЛОНГАЦИИ
# создадим словарь coef_sum_dict, в котором по ключу месяца будет сумма отгрузки за прошлый месяц
coef_sum_dict = {}

for last_month in range(2, 14) :
    goal_month = last_months_data.columns[last_month + 1]
    coef_sum = last_months_data[last_months_data.columns[last_month]].sum()
    #print(f'целевой месяц {goal_month}, а сумма будет из месяца {last_month} и равна {coef_sum}')
    coef_sum_dict[goal_month] = coef_sum

In [21]:
# 1-ЫЙ КОЭФ ПРОЛОНГАЦИИ
first_coef = first_prolongation.copy()

for col in first_coef.columns[1:13] :
    for string in first_coef.index:
        first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\384861441.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.186' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\384861441.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.257' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef.at[string, col] = round( first_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\384861441.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.247' has dtype incompatible with int64, p

In [ ]:
# ПРОБА СОХРАНЕНИЯ В EXCEL
first_coef.to_excel(excel_writer = 'C:\\IT\Report_about_prolongation.xlsx', sheet_name = 'Весь отдел', index=False)

In [22]:
# 2-ОЙ КОЭФ ПРОЛОНГАЦИИ
second_coef = second_prolongation.copy()

for col in second_coef.columns[1:13] :
    for string in second_coef.index:
        second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\1738367847.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.012' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\1738367847.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.03' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  second_coef.at[string, col] = round( second_coef.loc[string, col] / coef_sum_dict[col] , 3)
C:\Users\azott\AppData\Local\Temp\ipykernel_12560\1738367847.py:6: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.101' has dtype incompatible with in

In [ ]:
# ОБЩИЙ КОЭФ ПРОЛОНГАЦИИ ПО ОТДЕЛУ
# 1-ый коэф по отделу
first_coef_dept = first_prolongation.copy()
first_coef_dept = first_coef_dept.drop(['Account'], axis=1)
first_coef_dept = first_coef_dept.sum()
for col in first_coef_dept.index:
    first_coef_dept[col] = first_coef_dept[col] / coef_sum_dict[col]
    
second_coef_dept = second_prolongation.copy()
first_cosecond_coef_deptef_dept = second_coef_dept.drop(['Account'], axis=1)
second_coef_dept = second_coef_dept.sum()
for col in second_coef_dept.index:
    second_coef_dept[col] = second_coef_dept[col] / coef_sum_dict[col]

C:\Users\azott\AppData\Local\Temp\ipykernel_12560\519431742.py:8: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.45252293787343034' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  first_coef_dept[col] = first_coef_dept[col] / coef_sum_dict[col]


In [ ]:
# НЕ ЗАПУСКАТЬ, проверил на пропуски
for elem in general_table['month']:
    if elem is np.nan:
        print('есть')

есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть
есть


НОВАЯ АЛЬТЕРНАТИВА

In [ ]:
# ЛАСТ-МОНФС ИЗ ПЭЙ-ДАТА
# сразу после создания close_month_data
last_months_data = pay_data.copy()

end_project_list = []
# идём по каждому месяцу в числовом формате
for col in range(1, 17) :
    # внутри каждого месяца идём по каждому индексу, и заодно сразу находим id проекта
    for string in last_months_data.index :
        id_project = last_months_data.loc[string, 'id']
        if id_project in end_project_list :
            last_months_data.iat[string, col] = 0
        else:
            # по id проекта находим строку с информацией о последних месяцах этого проекта
            end_index = end_data[end_data['id'] == id_project].index
            if (end_data.iloc[end_index, col]).bool() == True :
                end_project_list.append(id_project)
                last_months_data.iat[string, col] = 0
                continue
            close_month_index = close_month_data[close_month_data['id'] == id_project].index
            if (close_month_data.iloc[close_month_index, col]).bool() == False :
                last_months_data.iat[string, col] = 0
            else:
                zero_index = zero_data[zero_data['id'] == id_project].index
                if (zero_data.iloc[zero_index, col]).bool() == True :
                    pay_index = pay_data[pay_data['id'] == id_project].index
                    last_months_data.iat[string, col] = pay_data.iloc[pay_index, (col-1)]